In [ ]:
import numpy as np
import pandas as pd
from config import *

In [ ]:
def harmonize_output(df):
    df = df.copy()

    # datetimes
    df["reference_date"] = pd.to_datetime(df["reference_date"], errors="coerce")
    df["target_end_date"] = pd.to_datetime(df["target_end_date"], errors="coerce")

    # strings
    df["target"] = df["target"].astype("string")
    df["location"] = df["location"].astype("string")
    df["output_type"] = df["output_type"].astype("string")
    df["output_type_id"] = df["output_type_id"].astype("string")

    # numeric
    # if you want strict integers with missing allowed:
    df["horizon"] = df["horizon"].astype("Int64")   # nullable integer
    df["value"] = df["value"].astype("float64")

    return df

In [ ]:
nowcast_file = f"{data_dir}nowcast_{epiyear}_{epiweek}.csv"
df_nowcast_pred = load_and_format_nowcast_pred(nowcast_file, ref_date, loc_name2loc)
df_nowcast_pred = harmonize_output(df_nowcast_pred)
# df_nowcast_pred.info()
df_nowcast_pred

In [ ]:
df_peak_pred = pd.read_csv(f"{results_dir}peak_forecasts/{format(ref_date,'%Y-%m-%d')}.csv")
df_peak_pred = parse_date_column(df_peak_pred,"reference_date")
df_peak_pred = harmonize_output(df_peak_pred)
# df_peak_pred.info()
df_peak_pred

In [ ]:
ensemble_name = 'WIS_Weighted_CU_Ensemble'
df_weekly_pred = load_pred_result_files(results_dir, ensemble_name, locations)
df_weekly_pred = df_weekly_pred[df_weekly_pred['reference_date']==ref_date]
df_weekly_pred = harmonize_output(df_weekly_pred)
# df_weekly_pred.info()
df_weekly_pred

In [ ]:
ensemble_name = 'WIS_Weighted_CU_Ensemble_edv'
df_weekly_pred_edv = load_pred_result_files(results_dir, ensemble_name, locations)
df_weekly_pred_edv = df_weekly_pred_edv[df_weekly_pred_edv['reference_date']==ref_date]
df_weekly_pred_edv["value"] = np.round(df_weekly_pred_edv["value"]/df_weekly_pred_edv["location"].map(pop_per_loc)*1e3,4)
df_weekly_pred_edv = harmonize_output(df_weekly_pred_edv)
df_weekly_pred_edv = df_weekly_pred_edv[df_weekly_pred_edv["target"]=='wk inc flu hosp']
df_weekly_pred_edv['target'] = 'wk inc flu prop ed visits'
# df_weekly_pred_edv.info()
df_weekly_pred_edv

In [ ]:
df_all = pd.concat([df_weekly_pred, df_nowcast_pred], ignore_index=True)
df_all = df_all.sort_values(
    by=["target", "location", "horizon"],
    ascending=[False, True, True],
    kind="mergesort"   # stable sort, preserves order within ties if you care
).reset_index(drop=True)
df_all = pd.concat([df_all, df_weekly_pred_edv, df_weekly_pred_edv, df_peak_pred], ignore_index=True)
df_all.to_csv(f"{results_dir}CU-ensemble/{format(ref_date,'%Y-%m-%d')}-CU-ensemble.csv", index=False, na_rep='NA')